# CHD-VLM Evaluation — Analysis & Paper Figures

**Notebook 3 of 3** — Full statistical analysis and publication-quality figures.

**Input**: `results/combined_all_models.csv` (from notebooks 01 + 02)  
**Outputs**: 
- Table 1: Per-condition metrics (Accuracy, Macro-F1, Kappa + 95% CI)
- Table 2: Per-class sensitivity/specificity for best condition per model
- Table 3: API cost summary
- Figures 2–5 and supplementary (PDF format, 300 DPI)

No GPU required.

## 0. Setup

In [ ]:
import os

REPO_DIR = "/content/chd-eval"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/rushankgoyal/chd-eval.git {REPO_DIR}

%cd {REPO_DIR}/new

!pip install -q pandas numpy scikit-learn scipy matplotlib seaborn pyyaml

import sys
sys.path.insert(0, "/content/chd-eval/new")
print("Setup complete.")

## 1. Load Results

In [ ]:
import pandas as pd
from pathlib import Path

# Upload your combined results CSV
# Option A: Upload manually
# from google.colab import files
# uploaded = files.upload()
# RESULTS_CSV = list(uploaded.keys())[0]

# Option B: Load from Drive
# from google.colab import drive
# drive.mount('/content/drive')
# RESULTS_CSV = "/content/drive/MyDrive/combined_all_models.csv"

# Option C: Already present in results/
RESULTS_CSV = "results/combined_all_models.csv"  # <-- EDIT THIS

df = pd.read_csv(RESULTS_CSV)
print(f"Loaded: {len(df)} rows")
print(f"Columns: {list(df.columns)}")
print(f"\nModels: {sorted(df['model_name'].unique())}")
print(f"Prompts: {sorted(df['prompt_id'].unique())}")
print(f"\nClass distribution:")
print(df[df['model_name'] == df['model_name'].iloc[0]]['true_label'].value_counts().to_string())

## 2. Full Statistical Analysis

In [ ]:
from src.analysis.analyze import run_full_analysis, print_summary

print("Running full analysis (bootstrap N=2000)...")
analysis = run_full_analysis(df, n_bootstrap=2000)

print_summary(analysis)

## 3. Table 1 — Per-Condition Classification Metrics

In [ ]:
metrics = analysis["metrics"]
ci_df = analysis["bootstrap_cis"]

# Merge CIs into metrics
table1 = metrics.merge(
    ci_df[["model_name", "prompt_id",
           "macro_f1_point", "macro_f1_ci_lower", "macro_f1_ci_upper",
           "accuracy_point", "accuracy_ci_lower", "accuracy_ci_upper"]],
    on=["model_name", "prompt_id"], how="left"
)

# Format for display
display_cols = ["model_name", "prompt_id", "n_total", "parse_rate",
                "accuracy", "accuracy_ci_lower", "accuracy_ci_upper",
                "macro_f1", "macro_f1_ci_lower", "macro_f1_ci_upper",
                "kappa"]
display_cols = [c for c in display_cols if c in table1.columns]
print("\n=== TABLE 1: Per-Condition Classification Metrics ===")
print(table1[display_cols].sort_values(["model_name", "prompt_id"]).to_string(index=False))

# Save
table1.to_csv("results/table1_metrics.csv", index=False)
print("\nSaved: results/table1_metrics.csv")

## 4. Table 2 — Per-Class Breakdown (Best Condition per Model)

In [ ]:
per_class = analysis["per_class_metrics"]

# For each model, find the best prompt (highest macro-F1)
best_conditions = (
    metrics.loc[metrics.groupby("model_name")["macro_f1"].idxmax()]
    [["model_name", "prompt_id"]]
)

table2_rows = []
for _, row in best_conditions.iterrows():
    mask = (
        (per_class["model_name"] == row["model_name"]) &
        (per_class["prompt_id"] == row["prompt_id"])
    )
    subset = per_class[mask].copy()
    table2_rows.append(subset)

table2 = pd.concat(table2_rows, ignore_index=True)
print("\n=== TABLE 2: Per-Class Metrics (Best Condition per Model) ===")
print(table2[["model_name", "prompt_id", "class", "f1", "sensitivity", "specificity"]].to_string(index=False))

table2.to_csv("results/table2_per_class.csv", index=False)
print("\nSaved: results/table2_per_class.csv")

## 5. Table 3 — API Cost Summary

In [ ]:
from src.analysis.analyze import cost_summary

cost_df = cost_summary(df)
if not cost_df.empty:
    print("\n=== TABLE 3: API Cost Summary ===")
    print(cost_df.to_string(index=False))
    print(f"\nTotal API cost: ${cost_df['total_cost_usd'].sum():.4f}")
    cost_df.to_csv("results/table3_api_cost.csv", index=False)
    print("Saved: results/table3_api_cost.csv")
else:
    print("No API cost data in results (HF-only results, or cost_usd column absent).")

## 6. McNemar Pairwise Significance Tests

In [ ]:
mcnemar = analysis["mcnemar_results"]
print(f"Total pairwise comparisons: {len(mcnemar)}")
print(f"Significant (BH p<0.05): {mcnemar['significant_05'].sum()}")
print(f"Significant (BH p<0.01): {mcnemar['significant_01'].sum()}")

sig_pairs = mcnemar[mcnemar["significant_05"]]
if not sig_pairs.empty:
    print("\n=== Significantly Different Pairs ===")
    print(sig_pairs[["condition_a", "condition_b", "n_shared", "p_bh"]].to_string(index=False))

mcnemar.to_csv("results/supplementary_mcnemar.csv", index=False)
print("\nSaved: results/supplementary_mcnemar.csv")

## 7. Generate All Figures

In [ ]:
from src.visualization.visualize import save_all_figures
from src.analysis.analyze import cost_summary
import matplotlib
matplotlib.rcParams["figure.dpi"] = 100  # lower for Colab display

cost_df = cost_summary(df)

save_all_figures(
    analysis,
    output_dir="figures",
    results_df=df,
    cost_df=cost_df if not cost_df.empty else None,
    fmt="pdf",
)

In [ ]:
# Display the holistic dashboard inline
import matplotlib.pyplot as plt
from src.visualization.visualize import plot_holistic_dashboard

fig = plot_holistic_dashboard(analysis, results_df=df)
plt.tight_layout()
plt.show()

In [ ]:
# Zip and download all figures + results tables
import shutil
from google.colab import files

shutil.make_archive("chd_vlm_outputs", "zip", ".", "figures")
shutil.make_archive("chd_vlm_tables", "zip", ".", "results")

files.download("chd_vlm_outputs.zip")
files.download("chd_vlm_tables.zip")
print("Downloaded figures and tables.")